### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [13]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from datetime import date, datetime

load_dotenv(override=True)

True

In [14]:
from accounts import Account

In [15]:
account = Account.get("Ed")
account

Account(name='ed', balance=378.6757599999993, strategy='You are a day trader that aggressively buys and sells shares based on news and market conditions.', holdings={'QS': 100, 'NVDA': 50, 'NVTS': 10, 'QUBT': 10, 'PONY': 10, 'QBTS': 10, 'OKLO': 10, 'AMZN': 3}, transactions=[100 shares of QS at 4.33866 each., 50 shares of NVDA at 148.19580000000002 each., 10 shares of NVTS at 7.3647 each., 10 shares of QUBT at 17.56005 each., 10 shares of PONY at 13.28652 each., 10 shares of QBTS at 14.99994 each., 10 shares of OKLO at 60.83142 each., 3 shares of AMZN at 212.41398 each.], portfolio_value_time_series=[('2025-06-25 15:43:48', 10000.0), ('2025-06-25 15:45:42', 10000.0), ('2025-06-25 15:46:58', 9999.134), ('2025-06-25 15:46:58', 9984.344), ('2025-06-25 15:49:14', 9984.344), ('2025-06-25 16:01:57', 9984.344), ('2025-06-25 16:02:27', 9984.197), ('2025-06-25 16:02:27', 9983.8465), ('2025-06-25 16:02:27', 9983.5813), ('2025-06-25 16:02:27', 9983.2819), ('2025-06-25 16:02:27', 9982.067700000001)

In [16]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

ValueError: Insufficient funds to buy shares.

In [5]:
account.report()

'{"name": "ed", "balance": 9915.832, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 28.056, "timestamp": "2025-06-25 14:12:45", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-06-25 14:12:45", 10026.832], ["2025-06-25 14:12:54", 9975.832]], "total_portfolio_value": 9975.832, "total_profit_loss": -24.167999999999665}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 28.056,
  'timestamp': '2025-06-25 14:12:45',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [10]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your account balance is $9,915.83. Your current holdings include 3 shares of Amazon (AMZN). Is there anything specific you would like to do with your account?

### Now let's build our own MCP Client

In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

In [12]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Ed, your account balance is $9,915.83. If you have any other questions or would like to make an investment, just let me know!

In [13]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9915.832, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 28.056, "timestamp": "2025-06-25 14:12:45", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-06-25 14:12:45", 10026.832], ["2025-06-25 14:12:54", 9975.832], ["2025-06-25 14:25:27", 10026.832]], "total_portfolio_value": 10026.832, "total_profit_loss": 26.832000000000335}


In [14]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9915.832, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 28.056, "timestamp": "2025-06-25 14:12:45", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-06-25 14:12:45", 10026.832], ["2025-06-25 14:12:54", 9975.832], ["2025-06-25 14:25:27", 10026.832], ["2025-06-25 14:25:43", 9939.832]], "total_portfolio_value": 9939.832, "total_profit_loss": -60.167999999999665}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Model Context Protocol Practice By Lee McCormick
# Lab2 : MCP Client + MCP Server

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from accounts import Account
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

load_dotenv(override=True)

# Demo Account from account module
account = Account.get("Ed")
account

account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")
account.report()
account.list_transactions()

# Write an MCP Server
params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

# Create an agent to mangage account using mcp
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


# Build MCP Client
mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

context = await read_accounts_resource("ed")
print(context)

Account.get("ed").report()

Ed, your current cash balance is approximately $378.68. Your holdings include:
- 100 shares of QS
- 50 shares of NVDA
- 10 shares of NVTS
- 10 shares of QUBT
- 10 shares of PONY
- 10 shares of QBTS
- 10 shares of OKLO
- 3 shares of AMZN

Is there anything specific you would like to do with your account?

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

Ed, your current account balance is approximately $378.68. If you need more information or want to perform any transactions, just let me know!

{"name": "ed", "balance": 378.6757599999993, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"QS": 100, "NVDA": 50, "NVTS": 10, "QUBT": 10, "PONY": 10, "QBTS": 10, "OKLO": 10, "AMZN": 3}, "transactions": [{"symbol": "QS", "quantity": 100, "price": 4.33866, "timestamp": "2025-06-25 15:46:58", "rationale": "High intraday jump (+31%) with significant volume, strong day trading candidate."}, {"symbol": "NVDA", "quantity": 50, "price": 148.19580000000002, "timestamp": "2025-06-25 15:46:58", "rationale": "Strong performance with high volume and industry interest, promising for day trading."}, {"symbol": "NVTS", "quantity": 10, "price": 7.3647, "timestamp": "2025-06-25 16:02:27", "rationale": "High volatility and daily movement make this a strong candidate for day trading."}, {"symbol": "QUBT", "quantity": 10, "price": 17.56005, "timestamp": "2025-06-25 16:02:27", "rationale": "Significant daily movement positions i

'{"name": "ed", "balance": 378.6757599999993, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"QS": 100, "NVDA": 50, "NVTS": 10, "QUBT": 10, "PONY": 10, "QBTS": 10, "OKLO": 10, "AMZN": 3}, "transactions": [{"symbol": "QS", "quantity": 100, "price": 4.33866, "timestamp": "2025-06-25 15:46:58", "rationale": "High intraday jump (+31%) with significant volume, strong day trading candidate."}, {"symbol": "NVDA", "quantity": 50, "price": 148.19580000000002, "timestamp": "2025-06-25 15:46:58", "rationale": "Strong performance with high volume and industry interest, promising for day trading."}, {"symbol": "NVTS", "quantity": 10, "price": 7.3647, "timestamp": "2025-06-25 16:02:27", "rationale": "High volatility and daily movement make this a strong candidate for day trading."}, {"symbol": "QUBT", "quantity": 10, "price": 17.56005, "timestamp": "2025-06-25 16:02:27", "rationale": "Significant daily movement positions 

In [ ]:
# Model Context Protocol Practice By Lee McCormick
# Lab2 : MCP Client + MCP Server

#### Exercise --> Make your own MCP Server! Make a simple function to return the current Date, 
# and expose it as a tool so that an Agent can tell you today's date.
# Harder optional exercise: then make an MCP Client, 
# and use a native OpenAI call (without the Agents SDK) to use your tool via your client.

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
from time_client import list_time_tools, get_time_tools_openai
from openai import OpenAI
from datetime import date, datetime

load_dotenv(override=True)

# ✅ MCP Server
params = {"command": "uv", "args": ["run", "time_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()
print("🛠️ Server - mcp_tools :", mcp_tools)

instructions = "You are able to get current date and time, and answer questions about the current date and time."
request = "What day is today?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="time_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("time_manager_server"):
        result = await Runner.run(agent, request)
    display(Markdown("#MCP SERVER"))
    display(Markdown(result.final_output))
    display(Markdown("###################################################"))

# ✅ MCP Client
mcp_tools = await list_time_tools()
print("🛠️ Cleint - mcp_tools :", mcp_tools)
openai_tools = await get_time_tools_openai()
print("🛠️ Cleint - openai_tools :", openai_tools)

request = "My name is Lee. What time is it?"

with trace("time_mcp_client"):
    agent = Agent(name="time_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown("#MCP Client"))
    display(Markdown(result.final_output))
    display(Markdown("###################################################"))

# ✅ OpenAI call (without the Agents SDK) to use your tool via your client.
openai = OpenAI()

# Prepare Messages for OpenAI
instructions = "You are able to get current date and time, and answer questions about the current date and time."
request = "What day is today? and What time is it?"

# Prepare Tools for OpenAI
def current_time_details() -> datetime:
    return datetime.now()

def current_date_details() -> date:
    return date.today()

# Messages and Tools
messages = [{"role": "system", "content": instructions}] + [{"role": "user", "content": request}]
tools = [
    {
        "type": "function",
        "function": {
            "name": "current_time_details",
            "description": "Get the current time in HH:MM:SS format",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "current_date_details",
            "description": "Get the current date in YYYY-MM-DD format",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

# Using OpenAI Completions with Tools
response = None
with trace("time_openai_chat_completion"):
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

# Handle the response and potential function calls
if response.choices[0].message.tool_calls:
    # Add the assistant's message to the conversation
    messages.append(response.choices[0].message)
    
    # Execute the function calls
    for tool_call in response.choices[0].message.tool_calls:
        function_name = tool_call.function.name
        
        if function_name == "current_time_details":
            result = current_time_details()
            function_response = result.strftime("%H:%M:%S")
        elif function_name == "current_date_details":
            result = current_date_details()
            function_response = result.strftime("%Y-%m-%d")
        else:
            function_response = "Function not found"
        
        # Add the function response to the conversation
        messages.append({
            "role": "tool",
            "content": function_response,
            "tool_call_id": tool_call.id
        })
    
    # Get the final response from the model
    final_response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    display(Markdown("# OPEN AI"))
    display(Markdown(final_response.choices[0].message.content))
else:
    display(Markdown("# OPEN AI"))
    display(Markdown(response.choices[0].message.content))

display(Markdown("###################################################"))

🛠️ Server - mcp_tools : [Tool(name='get_today', description='', inputSchema={'properties': {}, 'title': 'get_todayArguments', 'type': 'object'}, annotations=None), Tool(name='get_current_time', description='', inputSchema={'properties': {}, 'title': 'get_current_timeArguments', 'type': 'object'}, annotations=None)]


#MCP SERVER

Today is June 27, 2025.

###################################################

🛠️ Cleint - mcp_tools : [Tool(name='get_today', description='', inputSchema={'properties': {}, 'title': 'get_todayArguments', 'type': 'object'}, annotations=None), Tool(name='get_current_time', description='', inputSchema={'properties': {}, 'title': 'get_current_timeArguments', 'type': 'object'}, annotations=None)]
🛠️ Cleint - openai_tools : [FunctionTool(name='get_today', description='', params_json_schema={'properties': {}, 'title': 'get_todayArguments', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function get_time_tools_openai.<locals>.make_tool_invoker.<locals>.<lambda> at 0x119722de0>, strict_json_schema=True), FunctionTool(name='get_current_time', description='', params_json_schema={'properties': {}, 'title': 'get_current_timeArguments', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function get_time_tools_openai.<locals>.make_tool_invoker.<locals>.<lambda> at 0x1197223e0>, strict_json_schema=True)]


...

Hello Lee! The current time is approximately 17:13 (5:13 PM). Is there anything else you would like to know?

###################################################

# OPEN AI

Today is June 27, 2025, and the current time is 17:13:17.

###################################################